In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(
    "../data/raw/fraudTrain.csv",
    index_col=0,
    parse_dates=["trans_date_trans_time"],
)

In [2]:
df.shape, df.dtypes

((1296675, 22),
 trans_date_trans_time    datetime64[us]
 cc_num                            int64
 merchant                            str
 category                            str
 amt                             float64
 first                               str
 last                                str
 gender                              str
 street                              str
 city                                str
 state                               str
 zip                               int64
 lat                             float64
 long                            float64
 city_pop                          int64
 job                                 str
 dob                                 str
 trans_num                           str
 unix_time                         int64
 merch_lat                       float64
 merch_long                      float64
 is_fraud                          int64
 dtype: object)

Ajouter les observations après chaque run ici

In [3]:
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [4]:
df["is_fraud"].value_counts(normalize=True) * 100

is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64

Finding:

Wen can clearly see that there are only 0.579% of the transactions in the dataset that are fraudulent.

Implication:

These data show that only a very small proportion of the transactions are fraudulent. This means that it would be possible to train a model on this dataset that classifies all transactions as non-fraudulent and still achieve an accuracy of 99.42%, making it appear to perform very well. However, accuracy is not the same as precision, which is a metric defined as TP / (TP + FP). This is a major issue, as such a model would fail to identify any fraudulent transactions.

Decision:

Instead, I'm going to use a combination of the recall (proportion of real fraudulent transactions that the model detected) and precision metrics. To do that, I'm going to use the AUC-PR and determine a threshold that will minimize the total cost falsely detected fraudulent transactions and missed fraudulent transactions (fasle negative).   

In [5]:
print(df["is_fraud"].mean())

0.005788651743883394


In [6]:
print(df["trans_date_trans_time"].min())
print(df["trans_date_trans_time"].max())

df["month"] = df["trans_date_trans_time"].dt.to_period("M")

monthly = df.groupby("month")["is_fraud"].agg(number_of_fraud="sum", fraud_rate="mean", total_transactions="count")

print(monthly)

2019-01-01 00:00:18
2020-06-21 12:13:37
         number_of_fraud  fraud_rate  total_transactions
month                                                   
2019-01              506    0.009634               52525
2019-02              517    0.010368               49866
2019-03              494    0.006964               70939
2019-04              376    0.005523               68078
2019-05              408    0.005625               72532
2019-06              354    0.004113               86064
2019-07              331    0.003822               86596
2019-08              382    0.004373               87359
2019-09              418    0.005916               70652
2019-10              454    0.006603               68758
2019-11              388    0.005510               70421
2019-12              592    0.004197              141060
2020-01              343    0.006571               52202
2020-02              336    0.007031               47791
2020-03              444    0.006095            

Finding: 

December of 2019 has the highest number of transactions. Possibly due to the holiday period. 
The number of fraudulent transactions varies between 302 and 592 and the number of total transactions varies between 47791 and 141060 which is
a much bigger variation. December 2019 has the highest number of frauds (592) but has one of the lowest fraud rates. February 2019 has one of the lowest total transactions (49866) but has the highest fraud rate of the whole dataset. The number of frauds stays relatively stable, so the fraud rate varies because of the number of legitimate transactions. When total volume goes up (e.g., December), the rate goes down.

June 2020 only has 21 days worth of data. It is a partial month (20 days) so its monthly totals are not directly comparable to other months.

Implication:

What is shown by that is this variation in fraud rate could be related to the time of year. A seasonality. This could mean that the dataset for testing (2020) could share this relation between the time of year and fraud rate. Meaning that the 2020 dataset should have a similar seasonality (like that of the 2019 dataset) in fraud rate and total transactions. The model's decision threshold will have to be selected in a moment where it is most representative of the whole year. This is a form of data drift, so the threshold may need to be monitored and recalibrated over time.

Limitation: 

The validation period does not include a holiday season, while the test period does.

Decision: 

I will use a  time-based split: train on the earlier months of fraudTrain, validate on the late months, keep fraudTest untouched for the final evaluation.
